# Load the Titanic Dataset
This notebook loads the Titanic dataset from the data folder and displays preview, info, and statistics.

In [1]:
import pandas as pd

# Read CSV file
titanic = pd.read_csv('data/Titanic.csv')

# Show first 5 rows
print(titanic.head())

# Show dataset info
titanic.info()

# Statistical summary
print(titanic.describe())

   PassengerId  Survived  Pclass  \
0          892         0       3   
1          893         1       3   
2          894         0       2   
3          895         0       3   
4          896         1       3   

                                           Name     Sex   Age  SibSp  Parch  \
0                              Kelly, Mr. James    male  34.5      0      0   
1              Wilkes, Mrs. James (Ellen Needs)  female  47.0      1      0   
2                     Myles, Mr. Thomas Francis    male  62.0      0      0   
3                              Wirz, Mr. Albert    male  27.0      0      0   
4  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female  22.0      1      1   

    Ticket     Fare Cabin Embarked  
0   330911   7.8292   NaN        Q  
1   363272   7.0000   NaN        S  
2   240276   9.6875   NaN        Q  
3   315154   8.6625   NaN        S  
4  3101298  12.2875   NaN        S  
<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 co

## 2. Loading and Exploring Data
This section loads the Titanic dataset and checks for missing values.

In [2]:
# Count missing values in each column
print(titanic.isnull().sum())

# This helps identify columns with missing data, such as Age.

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


## 3. Cleaning Data
This section fills missing numeric values and removes unnecessary columns.

In [3]:
from sklearn.impute import SimpleImputer

# Create an imputer that fills missing values with the median
imputer = SimpleImputer(strategy='median')

# Select numeric columns
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']

# Fill missing values
titanic[numeric_cols] = imputer.fit_transform(titanic[numeric_cols])
print('Missing values:', titanic[numeric_cols].isnull().sum())

# Drop columns we don't need
titanic = titanic.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
print('Remaining columns:', list(titanic.columns))

Missing values: Age      0
Fare     0
SibSp    0
Parch    0
dtype: int64
Remaining columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


## 4. Scaling Features
Scaling puts all features on the same numeric scale. This helps models learn fairly.

In [4]:
from sklearn.preprocessing import StandardScaler

# Create a scaler object
scaler = StandardScaler()

# Numeric columns to scale
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']
X_numeric = titanic[numeric_cols]

# Fit and transform
X_scaled = scaler.fit_transform(X_numeric)
print('Mean of scaled data:', X_scaled.mean())
print('Std of scaled data:', X_scaled.std())

Mean of scaled data: 2.2841909119082167e-17
Std of scaled data: 1.0


## 5. Encoding Categorical Variables
Machines understand numbers. Convert categorical values into numeric form.

In [5]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Label Encoding for 'Sex'
label_encoder = LabelEncoder()
sex_encoded = label_encoder.fit_transform(titanic['Sex'])

print('Original:', list(titanic['Sex'].head()))
print('Encoded:', sex_encoded[:5])

# Show mapping
mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print('Mapping:', mapping)

# One-Hot Encoding for 'Embarked'
one_hot_encoder = OneHotEncoder(sparse_output=False, drop='first')
embarked_encoded = one_hot_encoder.fit_transform(titanic[['Embarked']])

print('Shape:', embarked_encoded.shape)
print('First 5 rows:')
print(embarked_encoded[:5])
print('Categories:', one_hot_encoder.categories_)

Original: ['male', 'female', 'male', 'male', 'female']
Encoded: [1 0 1 1 0]
Mapping: {'female': np.int64(0), 'male': np.int64(1)}
Shape: (418, 2)
First 5 rows:
[[1. 0.]
 [0. 1.]
 [1. 0.]
 [0. 1.]
 [0. 1.]]
Categories: [array(['C', 'Q', 'S'], dtype=object)]


## 6. Splitting Data Into Train and Test Sets
IMPORTANT: Always split before preprocessing in a full machine learning workflow.

In [6]:
from sklearn.model_selection import train_test_split

# Create features (X) and target (y)
X = titanic[['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']]
y = titanic['Survived']

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,  # 20% for testing
    random_state=42  # reproducibility
)

# Output
print('Training samples:', len(X_train))
print('Test samples:', len(X_test))
print('Split ratio:', len(X_train) / len(X) * 100, '%')

Training samples: 334
Test samples: 84
Split ratio: 79.90430622009569 %


## 7. Creating and Training Models
This section trains and evaluates three classification models on the Titanic data.

In [7]:
from sklearn.linear_model import LogisticRegression

# Model 1: Logistic Regression
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_accuracy = logistic_model.score(X_test, y_test)
print(f'Logistic Regression Accuracy: {logistic_accuracy:.4f}')

Logistic Regression Accuracy: 0.6190


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Model 2: Decision Tree
tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)
tree_train_score = tree_model.score(X_train, y_train)
tree_test_score = tree_model.score(X_test, y_test)
print(f'Training accuracy: {tree_train_score:.4f}')
print(f'Test accuracy: {tree_test_score:.4f}')

In [9]:
from sklearn.ensemble import RandomForestClassifier

# Model 3: Random Forest
forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
 )
forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)
forest_accuracy = forest_model.score(X_test, y_test)
print(f'Accuracy: {forest_accuracy:.4f}')

# Feature importance
for name, importance in zip(X.columns, forest_model.feature_importances_):
    print(f'{name}: {importance:.4f}')

Accuracy: 0.5833
Age: 0.3583
Fare: 0.4301
Pclass: 0.0512
SibSp: 0.0805
Parch: 0.0799


## 8. Evaluating Your Model
This section calculates accuracy, precision, recall, F1-score, the confusion matrix, and the classification report.

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Use the trained model for evaluation
model = forest_model
predictions = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, predictions)
print(f'Accuracy: {accuracy:.4f}')

# Precision, Recall, F1-Score
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

# Confusion Matrix
cm = confusion_matrix(y_test, predictions)
print(cm)

# Classification Report
print(classification_report(y_test, predictions))

Accuracy: 0.5833
Precision: 0.4737
Recall: 0.2647
F1-Score: 0.3396
[[40 10]
 [25  9]]
              precision    recall  f1-score   support

           0       0.62      0.80      0.70        50
           1       0.47      0.26      0.34        34

    accuracy                           0.58        84
   macro avg       0.54      0.53      0.52        84
weighted avg       0.56      0.58      0.55        84



## 9. Cross-Validation
This section evaluates the model across multiple splits for a more reliable accuracy estimate.

In [11]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier

# Create model
cv_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 5-fold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Get scores for each fold
scores = cross_val_score(cv_model, X, y, cv=kfold, scoring='accuracy')

print('Fold 1 accuracy:', scores[0])
print('Fold 2 accuracy:', scores[1])
print('Fold 3 accuracy:', scores[2])
print('Fold 4 accuracy:', scores[3])
print('Fold 5 accuracy:', scores[4])
print()
print(f'Mean accuracy: {scores.mean():.4f}')
print(f'Std dev: {scores.std():.4f}')

Fold 1 accuracy: 0.5833333333333334
Fold 2 accuracy: 0.6190476190476191
Fold 3 accuracy: 0.6190476190476191
Fold 4 accuracy: 0.6144578313253012
Fold 5 accuracy: 0.46987951807228917

Mean accuracy: 0.5812
Std dev: 0.0572


## 10. Hyperparameter Tuning (GridSearchCV)
This section searches for the best Random Forest settings using cross-validation.

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

# Base model
grid_model = RandomForestClassifier(random_state=42)

# Grid Search (5-fold cross-validation)
grid = GridSearchCV(
    estimator=grid_model,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1
 )

# Train
grid.fit(X_train, y_train)

# Results
print('Best Parameters:', grid.best_params_)
print(f'Best CV Score: {grid.best_score_:.4f}')
print(f'Test Score: {grid.score(X_test, y_test):.4f}')

Best Parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 50}
Best CV Score: 0.6586
Test Score: 0.6548
